In [1]:
from data_model_loader import *


model = load_model()
images = load_coco_2014_dataset()

# load file paths
with open('config.json', 'r') as file:
    config = json.load(file)
coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']
model_quantized_path = config["model_quantized_path"]

filename_to_image_id = get_image_ids()


OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Randomly selecting 1500 pictures ..
Done!


In [2]:
import torchvision.transforms as transforms

def preprocess_image(image):
    # convert greyscale to rgb
    if image.mode == 'L':
        image = image.convert('RGB')

    transform = transforms.Compose([
        transforms.ToTensor()  # Converts the image to a tensor [C, H, W] with values between 0 and 1
    ])
    
    image_tensor = transform(image)
    
    # Convert to uint8 and adjust tensor shape [H, W, C]
    image_tensor = (image_tensor * 255).byte()
    image_tensor = image_tensor.permute(1, 2, 0)  # [C, H, W] -> [H, W, C]
    
    # Add a batch dimension [1, H, W, C]
    image_tensor = image_tensor.unsqueeze(0)
    
    return image_tensor

In [3]:
import tensorflow as tf
from model import predict
import pathlib
from tqdm import tqdm
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

interpreter = tf.lite.Interpreter(model_path="model/ssd_mobilenet_quantized/ssd_mobilenet_int8.tflite")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = output_details[0]['shape']
print(input_details[0]['shape'])
interpreter.resize_tensor_input(input_details[0]["index"], (1, 1, 1, 3))

print(input_details[0]['shape'])

interpreter.allocate_tensors()
print(input_details[0]['shape'])

data_path = pathlib.Path(coco_folder)
results = []
for img in tqdm(images[:1], desc="Loading images"):
        # load image
        image_path = data_path/"val2014"/"val2014"/img
        image = Image.open(image_path)
        width, height = image.size

        image_tensor = preprocess_image(image)

        print(image_path)
        

[1 1 1 3]
[1 1 1 3]
[1 1 1 3]


Loading images: 100%|██████████| 1/1 [00:00<00:00, 47.63it/s]

data\coco2014\val2014\val2014\COCO_val2014_000000528311.jpg


In [4]:
input_details[0]

{'name': 'serving_default_input_tensor:0',
 'index': 0,
 'shape': array([1, 1, 1, 3]),
 'shape_signature': array([ 1, -1, -1,  3]),
 'dtype': numpy.uint8,
 'quantization': (0.0, 0),
 'quantization_parameters': {'scales': array([], dtype=float32),
  'zero_points': array([], dtype=int32),
  'quantized_dimension': 0},
 'sparsity_parameters': {}}

In [ ]:


def load_tflite_model(tflite_model_path):
    # Load the TFLite model and allocate tensors.
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    
    # Get input and output tensors.
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    return interpreter, input_details, output_details

def preprocess_image(image_path, input_details):
    # Open image and convert to RGB
    image = Image.open(image_path).convert('RGB')
    
    # Resize the image to the expected size
    target_shape = input_details[0]['shape'][1:3]  # height, width
    image = image.resize(target_shape)
    
    # Convert to numpy array and scale to [0, 255]
    image_np = np.array(image, dtype=np.uint8)
    
    # Add batch dimension [1, height, width, 3]
    image_np = np.expand_dims(image_np, axis=0)
    
    return image_np

def run_inference(interpreter, input_details, output_details, image):
    # Set the input tensor
    interpreter.set_tensor(input_details[0]['index'], image)
    
    # Run inference
    interpreter.invoke()
    
    # Get the output results
    output_data = {}
    for output_detail in output_details:
        output_data[output_detail['name']] = interpreter.get_tensor(output_detail['index'])
    
    return output_data


def process_output(output_data):
    num_detections = output_data['StatefulPartitionedCall:5']  # num_detections tensor
    detection_boxes = output_data['StatefulPartitionedCall:0'][0]      # detection_boxes tensor
    detection_classes = output_data['StatefulPartitionedCall:2'][0]  # detection_classes tensor
    detection_scores = output_data['StatefulPartitionedCall:2'][0]   # detection_scores tensor
    
    # Convert class index and score to a list of detections
    num_detections = num_detections[0]
    print(output_data['StatefulPartitionedCall:5'])

    
    """detections = []
    for i in range(num_detections):
        box = detection_boxes[i]  # [ymin, xmin, ymax, xmax]
        class_id = int(detection_classes[i]) + 1  # Class ID starts from 1
        score = detection_scores[i]
        detections.append({
            'box': box,
            'class_id': class_id,
            'score': score
        })
    
    return detections"""



tflite_model_path = "model/ssd_mobilenet_quantized/ssd_mobilenet_float32.tflite"
image_path = r"data\coco2014\val2014\val2014\COCO_val2014_000000322029.jpg"

interpreter, input_details, output_details = load_tflite_model(tflite_model_path)
image_np = preprocess_image(image_path, input_details)

output_data = run_inference(interpreter, input_details, output_details, image_np)


detections = process_output(output_data)

[100.]


In [8]:
output_data

{'StatefulPartitionedCall:6': array([[[-9.8323692e-03,  5.6441687e-04,  4.1892581e-02,  3.5092913e-02],
         [-1.0471346e-02, -8.0547720e-02,  6.8331122e-02,  1.6212142e-01],
         [-6.2480062e-02, -2.7799180e-02,  2.1418935e-01,  8.8875301e-02],
         ...,
         [ 1.9614771e-01, -1.3405621e-02,  8.2933044e-01,  1.0137303e+00],
         [-5.8508515e-03,  2.0341018e-01,  1.0102913e+00,  8.1383216e-01],
         [ 5.5006862e-02,  4.1899443e-02,  9.9047005e-01,  9.7592103e-01]]],
       dtype=float32),
 'StatefulPartitionedCall:0': array([[1916., 1911., 1916., 1784., 1916., 1916., 1754., 1916., 1907.,
         1724., 1916., 1730., 1911., 1818., 1916., 1916., 1878., 1902.,
         1784., 1226., 1916., 1862., 1906., 1724., 1911., 1250., 1784.,
         1862., 1784., 1916., 1779., 1876., 1912., 1790., 1784., 1916.,
         1893., 1784., 1644., 1796., 1862., 1862., 1520., 1916., 1911.,
         1916., 1730., 1878., 1796., 1550., 1730., 1861., 1784., 1784.,
         1878., 1784.